# HSJ Adversarial Robustness + ART BinaryInputDetector (Tabular Example)

This notebook demonstrates the full ART pipeline:

1. Train a **sklearn** base classifier on tabular data.
2. Wrap it in `SklearnClassifier`.
3. Generate adversarial examples with **HopSkipJump (HSJ)**.
4. Build a detection dataset: clean vs adversarial.
5. Train a **tf.keras** detector wrapped in `KerasClassifier`.
6. Use `BinaryInputDetector` to evaluate detection performance (TPR/FPR).


In [9]:

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from art.estimators.classification import SklearnClassifier, KerasClassifier
from art.attacks.evasion import HopSkipJump
from art.defences.detector.evasion import BinaryInputDetector

from art.utils import to_categorical

from tensorflow import models, layers



ImportError: cannot import name 'check_and_transform_label_format' from 'art.utils' (C:\Users\Jan\AppData\Roaming\Python\Python313\site-packages\art\utils.py)

In [ ]:
# Adjust the path if the CSV is in a different location
df = pd.read_csv("dataset.csv")

print(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# We assume:
#   - 'anomaly' is the label (0/1)
#   - 'train' = 1 → training, 0 → test
#   - 'channel' is a string; we drop it for now
#   - 'segment' is just an ID; we drop it
target_col = "anomaly"

feature_cols = [c for c in df.columns if c not in ["anomaly", "train", "channel", "segment"]]

train_df = df[df["train"] == 1].copy()
test_df  = df[df["train"] == 0].copy()

X_train = train_df[feature_cols].to_numpy().astype(np.float32)
y_train = train_df[target_col].to_numpy().astype(int)

X_test  = test_df[feature_cols].to_numpy().astype(np.float32)
y_test  = test_df[target_col].to_numpy().astype(int)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape :", X_test.shape, y_test.shape)


In [ ]:
base_model = LogisticRegression(max_iter=1000)
base_model.fit(X_train, y_train)

y_pred_test = base_model.predict(X_test)
clean_test_acc = accuracy_score(y_test, y_pred_test)
print(f"Base model clean test accuracy: {clean_test_acc:.4f}")


In [ ]:
def make_art_base_classifier(sk_model, X_train):
    """
    Wraps a sklearn model so ART attacks can query it.
    """
    x_min = float(X_train.min())
    x_max = float(X_train.max())
    clip_values = (x_min, x_max)

    art_clf = SklearnClassifier(
        model=sk_model,
        clip_values=clip_values,
    )
    return art_clf, clip_values

def _predict_labels_art(art_clf, X):
    """
    Robust prediction: if ART returns probabilities (n, k), take argmax;
    if it returns labels (n,), use directly.
    """
    preds = np.asarray(art_clf.predict(X))
    if preds.ndim == 1:
        return preds.astype(int)
    return np.argmax(preds, axis=1)

art_clf, clip_values = make_art_base_classifier(base_model, X_train)
print("ART SklearnClassifier created with clip_values:", clip_values)


In [ ]:
def generate_hsj_adversarials(art_clf, X_test, y_test, max_samples=100, hsj_kwargs=None):
    """
    Generate HSJ adversarials on a subset of test data and report robustness.
    """
    if hsj_kwargs is None:
        hsj_kwargs = dict(
            max_iter=20,
            max_eval=10000,
            init_eval=100,
            init_size=10,
            targeted=False,
            norm=2,
        )

    # Take subset
    n = min(max_samples, len(X_test))
    X = X_test[:n].astype(np.float32)
    y = y_test[:n].astype(int)

    # Encode labels to 0..K-1 for ART
    classes, y_int = np.unique(y, return_inverse=True)
    y_enc = y_int

    hsj = HopSkipJump(classifier=art_clf, **hsj_kwargs)

    print(f"Generating HSJ adversarials for {n} samples...")
    X_adv = hsj.generate(x=X, y=y_enc)

    # Robustness metrics
    y_pred_clean = _predict_labels_art(art_clf, X)
    y_pred_adv   = _predict_labels_art(art_clf, X_adv)

    clean_acc = accuracy_score(y_enc, y_pred_clean)
    adv_acc   = accuracy_score(y_enc, y_pred_adv)
    acc_drop  = clean_acc - adv_acc

    print(f"Clean accuracy (subset): {clean_acc:.4f}")
    print(f"Adv   accuracy (subset): {adv_acc:.4f}")
    print(f"Accuracy drop         : {acc_drop:.4f}")

    return X, y_enc, X_adv, classes, clean_acc, adv_acc, acc_drop

X_clean_sub, y_sub, X_adv_sub, classes, clean_acc, adv_acc, acc_drop = \
    generate_hsj_adversarials(art_clf, X_test, y_test, max_samples=100)


In [ ]:
def build_detector_dataset(X_clean, X_adv):
    """
    Construct dataset for the detector: clean vs adversarial.
    """
    X_clean_det = X_clean
    X_adv_det   = X_adv

    X_det = np.vstack([X_clean_det, X_adv_det])
    y_det = np.concatenate([
        np.zeros(len(X_clean_det), dtype=int),  # 0 = clean
        np.ones(len(X_adv_det), dtype=int),     # 1 = adversarial
    ])

    return X_det, y_det, X_clean_det, X_adv_det

X_det, y_det, X_clean_det, X_adv_det = build_detector_dataset(X_clean_sub, X_adv_sub)
print("Detector dataset built:", X_det.shape, y_det.shape)


In [ ]:
def make_keras_detector(input_dim, clip_values):
    """
    Small MLP detector: predicts clean (0) vs adversarial (1).
    """
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(2, activation="softmax"),  # 2 classes: clean/adv
    ])

    det_art = KerasClassifier(
        model=model,
        clip_values=clip_values,
        nb_classes=2,
        input_shape=(input_dim,),
    )
    return det_art

input_dim = X_det.shape[1]
det_art = make_keras_detector(input_dim, clip_values)
print("Keras detector backend created with input_dim:", input_dim)


In [ ]:
def train_art_binary_input_detector(X_det, y_det, det_art, nb_epochs=10, batch_size=32):
    """
    Train ART's BinaryInputDetector on clean+adv dataset.
    """
    detector = BinaryInputDetector(detector=det_art)
    detector.fit(X_det, y_det, nb_epochs=nb_epochs, batch_size=batch_size)
    return detector

detector = train_art_binary_input_detector(X_det, y_det, det_art, nb_epochs=10, batch_size=32)
print("BinaryInputDetector trained.")


In [ ]:
def evaluate_detector(detector, X_clean_det, X_adv_det):
    """
    Compute TPR (on adversarial) and FPR (on clean).
    """
    _, clean_flags = detector.detect(X_clean_det)
    _, adv_flags   = detector.detect(X_adv_det)

    det_fpr = float(clean_flags.mean())  # fraction of clean flagged
    det_tpr = float(adv_flags.mean())    # fraction of adv correctly flagged

    print(f"Detector TPR (adv caught):    {det_tpr:.4f}")
    print(f"Detector FPR (clean flagged): {det_fpr:.4f}")
    return det_tpr, det_fpr

det_tpr, det_fpr = evaluate_detector(detector, X_clean_det, X_adv_det)

results = pd.DataFrame([{
    "Model": "LogisticRegression (anomaly)",
    "CleanAccSubset": clean_acc,
    "AdvAccSubset": adv_acc,
    "AccDropSubset": acc_drop,
    "DetectorTPR": det_tpr,
    "DetectorFPR": det_fpr,
    "NumAttacked": len(X_clean_sub),
}])

display(results)
